Binary classification of jets(q,g) accuracy of 0.81 platue

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import roc_auc_score, confusion_matrix, precision_recall_fscore_support
import wandb
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load dataset ---
X = np.load("X_persistence.npy")
y = np.load("y_labels.npy")
print("Dataset size:", len(y))

# Reshape back to (N, 2, 40, 40) for CNN
N = X.shape[0]
X = X.reshape(N, 2, 40, 40).astype(np.float32)
y = y.astype(np.int64)

# Train/val split
train_size = int(0.8 * N)
val_size = N - train_size

dataset = TensorDataset(torch.tensor(X), torch.tensor(y))
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print(f"Train size: {train_size}, Val size: {val_size}")

# --- Define CNN Model ---
class PersistenceCNN(nn.Module):
    def __init__(self):
        super(PersistenceCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),   # (16, 20, 20)

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # (32, 10, 10)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # (64, 5, 5)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)  # binary classification
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

# --- Training setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PersistenceCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# --- Weights & Biases ---
wandb.init(
    project="persistence-cnn",
    name="three",   # run name
    mode="offline",   # no network calls
    settings=wandb.Settings(init_timeout=300),
    config={
        "learning_rate": 1e-3,
        "epochs": 100,
        "batch_size": 32,
        "architecture": "CNN+BatchNorm+Scheduler",
    }
)
config = wandb.config

# --- Training Loop ---
for epoch in range(config.epochs):
    model.train()
    train_loss, train_correct = 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        train_correct += (preds.argmax(1) == yb).sum().item()

    scheduler.step()

    # Validation
    model.eval()
    val_loss, val_correct = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)

            val_loss += loss.item() * xb.size(0)
            val_correct += (preds.argmax(1) == yb).sum().item()

            all_preds.extend(torch.softmax(preds, dim=1)[:,1].cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    # Metrics
    train_loss /= train_size
    train_acc = train_correct / train_size
    val_loss /= val_size
    val_acc = val_correct / val_size
    val_auc = roc_auc_score(all_labels, all_preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, np.array(all_preds) > 0.5, average="binary"
    )

    print(f"Epoch {epoch+1}: "
          f"Train Loss={train_loss:.4f}, Acc={train_acc:.4f} | "
          f"Val Loss={val_loss:.4f}, Acc={val_acc:.4f}, AUC={val_auc:.4f}, "
          f"Prec={precision:.4f}, Rec={recall:.4f}, F1={f1:.4f}")

    # Log metrics
    wandb.log({
        "epoch": epoch+1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_auc": val_auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "lr": scheduler.get_last_lr()[0],
    })

    # Confusion Matrix (every 5 epochs)
    if (epoch + 1) % 5 == 0:
        cm = confusion_matrix(all_labels, np.array(all_preds) > 0.5)
        fig, ax = plt.subplots(figsize=(4, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=["Quark", "Gluon"],
                    yticklabels=["Quark", "Gluon"], ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"Confusion Matrix (Epoch {epoch+1})")
        wandb.log({f"confusion_matrix_epoch{epoch+1}": wandb.Image(fig)})
        plt.close(fig)

wandb.finish()


Dataset size: 245109
Train size: 196087, Val size: 49022


Epoch 1: Train Loss=0.5806, Acc=0.6999 | Val Loss=0.5588, Acc=0.7186, AUC=0.7853, Prec=0.7410, Rec=0.6494, F1=0.6922
